---
title: Sampler examples
description: Practical examples of using the IBM Quantum Sampler primitive.
---


# Sampler examples

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.6.2
qiskit-ibm-runtime~=0.52.0
```
</AccordionItem>
</Accordion>

Generate entire error-mitigated quasi-probability distributions sampled from quantum circuit outputs. Leverage Sampler’s capabilities for search and classification algorithms like Grover’s and QVSM.

## Run a single experiment

Use Sampler to return the measurement outcome as bitstrings or counts of a single circuit.

In [1]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

sampler = Sampler(backend)
job = sampler.run([isa_circuit])
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]

print(f" > First ten results: {pub_result.data.meas.get_bitstrings()[:10]}")

 > First ten results: ['0001011110011001010110100110101101110011010110010110000001000110110010001000011000110100010010000110110111101000011000000000000', '0000111100011001100000111110001011110000001101111100110100000001001000110010001011100010110001100111111110100110100100001000001', '0001001100000000010011000110001111001111010100010111000001101011011110001001000010101000001010010000110000000010000001001000000', '0100110110010100010010100000111001000001101101011100000010010001001001100000001000000000001010100000100000101110001100111101000', '1111000100111001000110100100100001100111100100001100010001000100111011010011100100011000000000001000100000001000010100001000000', '0010000001010100011100000000000000000011110100101100000101100010101110100011100111000000100000000001000000000110011000000000001', '1011100100011011011001011100110101001011000000111010000011110000101000000001100110000100110010110010110010000000001000000100001', '11110001110010000011110110000111000100110110001001100001111

## Run multiple experiments in a single job

Use Sampler to return the measurement outcome as bitstrings or counts of multiple circuits in one job.

In [2]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
circuits = [iqp(mat) for mat in mats]
for circuit in circuits:
    circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits)
result = job.result()

for idx, pub_result in enumerate(result):
    print(
        f" > First five results for pub {idx}: "
        f"{pub_result.data.meas.get_bitstrings()[:5]}"
    )

 > First five results for pub 0: ['0101110010010001011111110110001110111100000001000000000010110000011110110000100001001100011001000110000010000000010100001001100', '0110001000100100010010100001010111111101100000010110100111010101000111010000100001000000011010011100101100010000010011100010110', '1001111110100000110110110100111110010111111001011000010101000000000010001000000001110010010001010010001000000000000100000110000', '1111001011000001000001000101100000101011000000000100011010001111001100110100110000010000000111011001001010010110000111000011100', '0010000100000110010110110011011010010000110011100001001100010100011100000010101010110000000110010001001011100001000000101100110']
 > First five results for pub 1: ['1111000000000100001101011101000101100000000100011100001001011010100011010010110000011101100000110111000010010001010011001010000', '0110110011000110001101101111100100100100100100000001100010000101001000000000011100000011010100000010000000100111000000011010000', '01010010110001

## Run parameterized circuits

Run several experiments in a single job, leveraging parameter values to increase circuit reusability.

In [3]:
import numpy as np
from qiskit.circuit.library import real_amplitudes
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

# Step 1: Map classical inputs to a quantum problem
circuit = real_amplitudes(num_qubits=n_qubits, reps=2)
circuit.measure_all()

# Define three sets of parameters for the circuit
rng = np.random.default_rng(1234)
parameter_values = [
    rng.uniform(-np.pi, np.pi, size=circuit.num_parameters) for _ in range(3)
]

# Step 2: Optimize problem for quantum execution.

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

# Step 3: Execute using IBM Quantum primitives.
sampler = Sampler(backend)
job = sampler.run([(isa_circuit, parameter_values)])
result = job.result()
# Get results for the first (and only) PUB
pub_result = result[0]
# Get counts from the classical register "meas".
print(
    f" >> First five results for the meas output register: "
    f"{pub_result.data.meas.get_bitstrings()[:5]}"
)

 >> First five results for the meas output register: ['0011010100010111100110001100100000110001110000100000011000100010000101001010101101011110100010000000010110100001001010111111001', '0100011000010000110100010100010111101010100000011100111011011111100111001110001111001100011110110110001001000001000100010101101', '0000011000010011010100001000001000100110100110001111100011101100110010011110111101110010100100000000001111111111110000001111111', '0010000111000000111010100001100000011110000001110010011111111011011001101101001111001110100000011100101100001000001110000000110', '0011011000010100010100100101100011101110010001010101011000011111111000101110110100000001010011010101111011111111001110111011101']


## Use batches and advanced options

Explore the batch [execution mode](/docs/ibm-quantum-compute/execution-modes) and advanced options to optimize circuit performance on QPUs.

In [4]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Batch, QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

rng = np.random.default_rng(1234)
mat = np.real(random_hermitian(n_qubits, seed=rng))
circuit = iqp(mat)
circuit.measure_all()
mat = np.real(random_hermitian(n_qubits, seed=rng))
another_circuit = iqp(mat)
another_circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
another_isa_circuit = pm.run(another_circuit)

# The context manager automatically closes the batch.
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    job = sampler.run([isa_circuit])
    another_job = sampler.run([another_isa_circuit])
    result = job.result()
    another_result = another_job.result()

# first job

print(
    f" > The first five measurement results of job 1: "
    f"{result[0].data.meas.get_bitstrings()[:5]}"
)

 > The first five measurement results of job 1: ['1010010010011101000111111001111011100011100101000000011011010110010000110000100001010100010100000000000101001111001000110001000', '0001000001011011110110100111010100000100010100010001010010110111111000000011001100101101000001100000001001000011100010101011001', '0011010101110101101011110100011000111000000010011000010001110011001110100000100010100000001010100000000000101000000000010000100', '1111011001000101110001010001010111101001000100101011010010000000011001100010000010011110101010000000100000101000001100111000100', '0000010101001101101001001010011000110000010100010100011110100001001010100010000110010110100001000011000100000001001010011100000']


In [5]:
# second job
print(
    " > The first five measurement results of job 2:",
    another_result[0].data.meas.get_bitstrings()[:5],
)

 > The first five measurement results of job 2: ['1001001000000001111010101000101000010000101010011010010011000000001000100101001001100000000011101001000100000000000110110100010', '1000011000100110111010100001110000000100000001000111010001000110001110001111011101000000101001011000001100100000000000001010000', '1110100100110111100001111000010010101100100001101010001000000111101100001000000010010000100000111001001001000000010000011011000', '0111100011101100010101011100000000000101001000000010011010000010000010100101110010010000010011010000000000000000110001100000000', '0001100001110011011000111000101111011001001011001100011100000010001100000011100011110001000001010100011011000000000101110000001']


## Next steps

<Admonition type="tip" title="Recommendations">
    - [Specify advanced runtime options](runtime-options-overview).
    - Practice with primitives by working through the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum&reg; Learning.
    - Learn how to transpile locally in the [Transpile](/docs/qiskit/transpile/) section.
    - Try the [Compare transpiler settings](/docs/qiskit/circuit-transpilation-settings) guide.
    - Understand the [Job limits](/docs/ibm-quantum-compute/job-limits) when sending a job to an IBM&reg; QPU.
</Admonition>